In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import build_final_ensembles
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA06_gm_selection import setup_AvgSA06_gcim_gm_selection


cfg = load_config()

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:15: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# AvgSA([0, 6])

In [3]:
# canonical Stage-1 output (consumed by the post-processing notebook)
final_ensembles_fp = cfg["proc_data"]["gm_selection"] / "AvgSA_06_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded)
GMS = cfg["proc_data"]["gm_selection"]
stage_fps = {
    "select": [
        GMS / "AvgSA_06_prelim_selection.pickle",        # round 1 selection (all sites)
        GMS / "AvgSA_06_prelim_reselection.pickle",      # round 2 reselection (no-ensemble sites)
        GMS / "AvgSA_06_prelim_reselection_rd03.pickle", # round 3 reselection (usually none)
    ],
    "optimise": [
        GMS / "AvgSA_06_optimised_selection_rd01.pickle",  # round 1 optimise
        GMS / "AvgSA_06_optimised_selection_rd02.pickle",  # round 2 optimise
        GMS / "AvgSA_06_optimised_selection_rd03.pickle",  # round 3 optimise
    ],
}

# inputs used for provenance fingerprinting (hashed by their file bytes)
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_06.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_data_file":  cfg["proc_data"]["site_hazard"] / "AvgSA_06_disagg_data_60sites.pickle",
    "disagg_stats_file": cfg["proc_data"]["site_hazard"] / "AvgSA_06_disagg_stats_60sites.pickle",
    "site_model_file":   cfg["hazard_models"]["eshm20_AvgSA_site_model_all"],
}

# Selection scheme: 3 rounds, dropping progressively more causal-parameter bounds.
# Round 2 reselects with only the distance bound free but optimises with distance +
# vs30 free (a dict spec), reproducing the legacy AvgSA_06 behaviour exactly.
round_unbounded = [
    [],                                            # round 1: all of m, d, vs30 bounded
    {"select": ["d"], "optimise": ["d", "vs30"]},  # round 2: reselect d-free; optimise d+vs30-free
    ["m", "d", "vs30"],                            # round 3: drop all bounds
]

# Per round: optimise the whole round work-set (True) or only sites still failing
# after that round's selection (False). [False, True, False] reproduces the legacy
# result (round 2 re-optimises the reselected sites).
force_optimisation = [False, True, False]

# other stuff:
rng_seed = 1

# Set True to rebuild every stage + the final artifact, ignoring (and overwriting)
# any existing cache. Leave False for normal runs: an input change then raises
# StaleCacheError instead of silently reusing stale results.
FORCE_RECOMPUTE = False

In [4]:
# set up the record selection
site_poe_disaggs, disagg_stats, site_model, basic_selection_ctx, gm_db = setup_AvgSA06_gcim_gm_selection()

# load the gcim distributions
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print("Existing GCIM distribution data loaded...")
else:
    print("No existing GCIM distribution data found...")

Existing GCIM distribution data loaded...


## Stage 1 - Build final ensembles (slow compute)

Runs the configurable 3-round selection + optimisation engine and saves the canonical
`AvgSA_06_final_ensembles.pickle` (+ a `.manifest.json` provenance sidecar). Round 2
uses a dict spec so reselection drops only the distance bound while optimisation drops
distance + vs30 - reproducing the legacy AvgSA_06 scheme. Each step is provenance-cached
(`StaleCacheError` on an input change; `FORCE_RECOMPUTE = True` to rebuild). Progress bars
are labelled `R{n}/3 select/optimise` and cached stages print `[cache] ... loaded`.

Post-processing lives in **`wp1pt3pt8g-gm_selection_AvgSA_06_stage2_postprocess.ipynb`**.

In [5]:
final_ensembles = build_final_ensembles(
    site_poe_disaggs,
    disagg_stats,
    gcim_dists,
    gm_db,
    basic_selection_ctx,
    site_model,
    source_fps=source_fps,
    stage_fps=stage_fps,
    output_fp=final_ensembles_fp,
    round_unbounded=round_unbounded,
    force_optimisation=force_optimisation,
    rng_seed=rng_seed,
    force_recompute=FORCE_RECOMPUTE,
)

-- Round 1/3 -- bounds all bounds | select: 360 sites
[cache] 'AvgSA_06_prelim_selection.pickle' loaded (inputs match).
Sub-Optimal! - Preliminary ensembles do not pass for 86 combinations of site and poe
No preliminary ensembles found for 2 combinations of site and poe
[cache] 'AvgSA_06_optimised_selection_rd01.pickle' loaded (inputs match).
Sub-Optimal! - Optimised ensembles do not pass for 27 combinations of site and poe
-- Round 2/3 -- select free: d / optimise free: d,vs30 | select: 2 sites | force-optimise all
[cache] 'AvgSA_06_prelim_reselection.pickle' loaded (inputs match).
Sub-Optimal! - Preliminary ensembles do not pass for 2 combinations of site and poe
[cache] 'AvgSA_06_optimised_selection_rd02.pickle' loaded (inputs match).
Sub-Optimal! - Optimised ensembles do not pass for 2 combinations of site and poe
-- Round 3/3 -- bounds free: m,d,vs30 | select: 0 sites
[cache] 'AvgSA_06_optimised_selection_rd03.pickle' loaded (inputs match).
OK! - Optimised ensembles pass for all c